# Оценка метрик на M2 пайплайне

In [1]:
import sys
import os
import json

sys.path.append('./../')

In [2]:
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from eval.self_repeatness import count_self_repeating_part
from eval.variarity import count_fields_variance
from eval.md_classes import MDField, DataStep

load_dotenv()

True

In [3]:
MODEL = 'gpt-5.4-nano'
MODEL_PROVIDER = 'openai'
TEMPERATURE = 0.0
COMPARER_SP = './../prompts/llm_judges/fields_comparer.txt'

STATS_DIR = './../data/stats/m2'
DF_PATH = './../data/md_generations/m2_synth_bio_mds.csv'
EMBEDS_PATH = './../data/md_generations/embeds/m2/'
QUERIES_CNT = 5
RUNS = 4

## Просто доп подгрузки для подсчета

In [4]:
from langchain_core.callbacks import BaseCallbackHandler


class TokenCounter(BaseCallbackHandler):
    def __init__(self):
        self.prompt_tokens     = 0
        self.completion_tokens = 0
        self.total_tokens      = 0
        self.n_calls           = 0

    def on_llm_end(self, response, **kwargs):
        usage = None

        # LangChain wraps different providers differently
        if hasattr(response, "llm_output") and response.llm_output:
            usage = response.llm_output.get("token_usage") or \
                    response.llm_output.get("usage")

        # OpenAI via langchain_openai exposes it here
        if usage is None and response.generations:
            for gen_list in response.generations:
                for gen in gen_list:
                    info = getattr(gen, "generation_info", None) or {}
                    usage = info.get("usage") or info.get("token_usage")
                    if usage:
                        break

        if usage:
            self.prompt_tokens     += usage.get("prompt_tokens",     0)
            self.completion_tokens += usage.get("completion_tokens", 0)
            self.total_tokens      += usage.get("total_tokens",      0)
        self.n_calls += 1

In [5]:
from typing import List

import time
from eval.judges.repeating.fields_repeating import CompareState, FieldsComparator

def read_prompts(pr_path:str) -> str:
    with open(pr_path, 'rt') as fio:
        res = fio.read()
    return res 

field_comparer = FieldsComparator(read_prompts(COMPARER_SP), MODEL, MODEL_PROVIDER, TEMPERATURE)

def factory_func(md_name: str, md_description: str):
    def llm_judge(target_field: str, candidates: List[str]) -> List[str]:
        is_complete = False

        while not is_complete:
            try:
                counter = TokenCounter()

                state = CompareState()
                state['md_schema_name'] = md_name
                state['md_schema_descr'] = md_description
                state['main_field_str'] = target_field
                state['potention_similar_fields'] = candidates
                
                wf = field_comparer.workflow
                res = wf.invoke(
                    state,
                    config={"callbacks": [counter]})
                print('Spend during last LLM call: ', str({
                    'prompt': counter.prompt_tokens,
                    'completion': counter.completion_tokens,
                    'total': counter.total_tokens
                }))

                if res['result'] != None and res['result'].same_fields_from_list != None:
                    is_complete = True
                print(res['result'].same_fields_from_list)
            except Exception as e:
                print(f'Catched \"{str(e)}\". Retrying in 120 sec')
                time.sleep(120)

        return res['result'].same_fields_from_list
    
    return llm_judge

## Запуск кода обхода таблицы и сохранения

### Проба сравнения

In [6]:
def field_converter(df: pd.DataFrame, embeds_folder:str) -> List[MDField]:
    fields = {}
    embeds = {}

    for idx, row in df.iterrows():
        if 'dag_step' in row.keys():
            block = DataStep(row['dag_step'], row['dag_descr'])
        else:
            block = None
        md_field = MDField(row['field_name'], row['field_description'], row['field_type'], block=block)
        fields.update({row['field_name']: md_field})

        embed = np.load(os.path.join(embeds_folder, f"{idx}.npy"))
        embeds.update({row['field_name']: embed})
    return fields, embeds


In [7]:
os.makedirs(STATS_DIR, exist_ok=True)
df = pd.read_csv(DF_PATH, index_col=[0])

for q_idx in range(QUERIES_CNT):
    queries_fields = df.loc[df['query_id'] == q_idx]

    md_name = queries_fields.iloc[0]['md_short_name']
    md_descr = queries_fields.iloc[0]['md_description']

    llm_judger = factory_func(md_name, md_descr)

    # collect per run stats
    for r in range(RUNS):
        df_qr_fields = queries_fields.loc[df['run_id'] == r]
        fields, embeds = field_converter(df_qr_fields, EMBEDS_PATH)

        volume = len(fields.keys())
        repeated_part = count_self_repeating_part(fields, embeds, llm_judger, 0.95, 0.5, 1)

        with open(os.path.join(STATS_DIR, f"q{q_idx}_r{r}_stat.json"), 'wt') as fo:
            json.dump({'q': q_idx, 'r': r, 'volume': volume, 'repeated_part': repeated_part}, fo)

    # collect per exp stats
    vars = np.zeros(shape=(RUNS, RUNS))
    ious = np.zeros(shape=(RUNS, RUNS))
    for i in range(0, RUNS - 1):
        df_fields_i = queries_fields.loc[df['run_id'] == i]
        fields_i, embeds_i = field_converter(df_fields_i, EMBEDS_PATH)
        
        for j in range(1 + 1, RUNS):
            df_fields_j = queries_fields.loc[df['run_id'] == j]
            fields_j, embeds_j = field_converter(df_fields_j, EMBEDS_PATH)
    
            results = count_fields_variance(fields_i, fields_j, embeds_i, embeds_j, llm_judger, 0.95, 0.5, 1)

            vars[i, j] = results['variance']
            vars[j, i] = results['variance']
            
            ious[i, j] = results['iou']
            ious[j, i] = results['iou']

    with open(os.path.join(STATS_DIR, f"q{q_idx}_stat.json"), 'wt') as fo:
        json.dump({'q': q_idx, 'variances': vars.tolist(), 'ious': ious.tolist()}, fo)

Spend during last LLM call:  {'prompt': 1959, 'completion': 302, 'total': 2261}
['tumor_histology']
Spend during last LLM call:  {'prompt': 2220, 'completion': 301, 'total': 2521}
[]
Spend during last LLM call:  {'prompt': 2035, 'completion': 319, 'total': 2354}
[]
Spend during last LLM call:  {'prompt': 1973, 'completion': 288, 'total': 2261}
[]
Spend during last LLM call:  {'prompt': 1664, 'completion': 315, 'total': 1979}
[]
Spend during last LLM call:  {'prompt': 1468, 'completion': 173, 'total': 1641}
[]
Spend during last LLM call:  {'prompt': 2758, 'completion': 355, 'total': 3113}
['clinical_linkage_key']
Spend during last LLM call:  {'prompt': 2147, 'completion': 476, 'total': 2623}
['patient_sample_id']
Spend during last LLM call:  {'prompt': 2136, 'completion': 287, 'total': 2423}
[]
Spend during last LLM call:  {'prompt': 2124, 'completion': 257, 'total': 2381}
[]
Spend during last LLM call:  {'prompt': 1672, 'completion': 290, 'total': 1962}
[]
Spend during last LLM call:  

KeyboardInterrupt: 

You are a metadata field comparer for scientific datasets.

You receive the user's query to generate dataset, metadata table name and description,
also ONE TARGET field and a LIST OF CANDIDATE fields.
Decide which candidates describe EXACTLY the same meaning, that a TARGET fields.

COMPARE the fields using next RULES:

1. BLOCK MISMATCH --> the fields are NOT the same.
  Sometimes TARGET and CANDIDATE fields carry a block label (like "[\"sample_preparation\" Data Workflow Step]"),
  not only the name, description and type. Before comparing fields, you must compare the blocks:
  if thier blocks are DIFFERENT (not only in names, but also in processes inside the block, thier descriptoins), 
  then they CAN'T be the same! Do not proceed the 2nd step in such situation. 
  If the blocks are nearly the same, or block of one field is a part of some other bigger
  block, than you can think about it as a block matching.

2. SUBJECT MISMATCH --> NOT equivalent.
  Identify the real-world subject of each field: the entity whose property is being recorded (e.g. the patient, the sample, the instrument,
  the treatment, the experiment session, the file).
  If the TARGET and a CANDIDATE describe a property of DIFFERENT subjects, they are NOT equivalent — even if their descriptions share vocabulary,
  examples, or allowed values.
  Skip Rule 3 for that candidate.

3. EXACTLY SAME PROPERTY OR MEASURMENT --> the fields ARE the same.
  A candidate is equivalent if ALL of the following hold:
    - No block conflict (no labels, or matching labels from Rule 1).
    - The fields are applied to the same subject or property.
    - Same real-world property in names AND units: the two fields describes the exactly property in provided research.
    - Compatible data types in (float vs int is fine; string vs float is not).

  These differences do NOT disqualify equivalence:
    - Different field names ("temp_c" vs "temperature_celsius")
    - Minor wording variation ("chamber temperature" vs "ambient temperature during imaging")
    - Different units for the same quantity ("_celsius" vs "_kelvin")

  If comparing fields have a different meanings in thier description - they are NOT the same (like voxel_size in images properties and in microscope). 
  To find out that details, you can use your reasoning powers - don't hesitate to reason well.

REASONING:
In your reasoning section, analyze each possible candidate field with all described rules below. After analyzing all of them - make decisions. 

OUTPUT: return the valid json only with reasoning block and the names of fields that are equivalent to the target.
If no candidate is equivalent, return an empty list.
The example:
{{
    "reasoning": "your reasoning",
    "same_fields_from_list": [*list of value(-s), or empty list*]
}}

In [ ]:


You are a metadata field comparer for scientific datasets.

You receive the user's query to generate dataset, metadata table name and description,
also ONE TARGET field and a LIST OF CANDIDATE fields.
Decide which candidates describe EXACTLY the same meaning, that a TARGET fields.

COMPARE the fields using next RULES:

1. BLOCK MISMATCH --> the fields are NOT the same.
  Sometimes TARGET and CANDIDATE fields carry a block label (like "[\"sample_preparation\" Data Workflow Step]"),
  not only the name, description and type. If thier block labels are DIFFERENT, they CAN'T be the same!
  Do not proceed the 2nd step in such situation.

2. SUBJECT MISMATCH --> NOT equivalent.
  Identify the real-world subject of each field: the entity whose property is being recorded (e.g. the patient, the sample, the instrument,
  the treatment, the experiment session, the file).
  If the TARGET and a CANDIDATE describe a property of DIFFERENT subjects, they are NOT equivalent — even if their descriptions share vocabulary,
  examples, or allowed values.
  Skip Rule 3 for that candidate.


3. EXACTLY SAME PROPERTY AND MEASURMENT --> the fields ARE the same.
  A candidate is equivalent if ALL of the following hold:
    - No block conflict (no labels, or matching labels from Rule 1).
    - The fields are applied to the same subject or property.
    - Same real-world property in names AND units: the two fields describes the exactly property in provided research.
    - Compatible data types in (float vs int is fine; string vs float is not).

  These differences do NOT disqualify equivalence:
    - Different field names ("temp_c" vs "temperature_celsius")
    - Minor wording variation ("chamber temperature" vs "ambient temperature during imaging")
    - Different units for the same quantity ("_celsius" vs "_kelvin")

  If comparing fields have a different meanings in thier description - they are NOT the same (like voxel_size in images properties and in microscope). 
  To find out that details, you can use your reasoning powers - don't hesitate to reason well.

OUTPUT: return the valid json only with reasoning block and the names of fields that are equivalent to the target.
If no candidate is equivalent, return an empty list.
The example:
{{
    "reasoning": "your reasoning",
    "same_fields_from_list": [*list of value(-s), or empty list*]
}}